In [1]:
# Installation de toutes les dépendances nécessaires
!pip install torch --index-url https://download.pytorch.org/whl/cu121
!pip install librosa soundfile audioread datasets "transformers[torch]" evaluate jiwer GPUtil psutil requests

Looking in indexes: https://download.pytorch.org/whl/cu121


In [7]:
import os
os.environ["DATASETS_AUDIO_BACKEND"] = "soundfile"

In [2]:
import csv
import torch
import evaluate
from datasets import Dataset, Audio, concatenate_datasets, load_dataset
from transformers import (
    WhisperProcessor, 
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback
)
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import numpy as np
from google.cloud import storage
import io
import tempfile
import requests
import tarfile
import soundfile as sf
import pandas as pd
from scipy.signal import resample
import psutil
import os
import GPUtil as GPU
import gc

#### CONFIGURATION VERTEX AI + GCS

In [9]:
# Configuration GCS
GCS_CONFIG = {
    'project_id': 'madameai-dev', 
    'bucket_name': 'madameai-training-dataset-audio',
    'tts_prefix': 'synthetic/',  # Dossier dans le bucket pour TTS
    'csv_file': 'synthetic/v1_text.csv',  # Chemin du CSV dans GCS
    'output_prefix': 'models/whisper-tts-commonvoice/',  # Où sauvegarder le modèle
}

In [10]:
# Accents disponibles
ACCENTS = ['hispanique', 'anglais', 'hindi']

## Documentation

https://docs.google.com/document/d/1CveeU1sHWyCkeoR33Gr2O5iMHmcS4ISXzN34ywHQPp8/edit?tab=t.0#heading=h.otl0otb6v2rj

In [11]:
# Configuration de l'entraînement
CONFIG = {
    'model_name': 'openai/whisper-small',
    
    # ===== OPTIMISATION =====
    'learning_rate': 1e-5,
    'weight_decay': 0.01,
    'warmup_steps': 500,
    
    # ===== ENTRAÎNEMENT =====
    'num_epochs': 5,
    'batch_size': 4,
    'gradient_accumulation_steps': 4,
    
    # ===== ARRÊT ANTICIPÉ =====
    'early_stopping_patience': 3,
    'load_best_model_at_end': True,
    'metric_for_best_model': 'wer',
    'greater_is_better': False,
    
    # ===== ÉVALUATION =====
    'eval_steps': 100,
    'save_steps': 100,
    'save_total_limit': 3,
    
    # ===== COMMON VOICE =====
    'use_common_voice': True,
    'common_voice_max_samples': None,  # Pas de limite — charge tout
    
    # ===== FRANCOFLEX =====
    'use_francoflex': True,
    'francoflex_max_samples': None,  # Pas de limite — charge tout
    
    # ===== DONNÉES UTILISATEURS =====
    'use_data_users': True,
    'data_users_csv': 'francoflex_raw_data/data_users.csv',
    'data_users_max_samples': None,  # Pas de limite — charge tout
    
    # ===== DONNÉES SCRAPPÉES =====
    'use_scraped': True,
    'data_scraped_csv': 'madameai-training-dataset-audio/scraped/public_videos_clips/scraped_public_videos_clips_directory.csv',
    'data_scraped_max_samples': None,  # Pas de limite — charge tout

    # ===== RÉGULARISATION =====
    'dropout': 0.1,
    'label_smoothing': 0.1,
}


## Initialisation du bucket (ne change pas)

In [12]:
def get_gcs_client():
    """Initialise le client GCS (authentification automatique sur Vertex AI)"""
    return storage.Client(project=GCS_CONFIG['project_id'])


def load_csv_from_gcs(bucket_name, csv_path):
    """Charge le CSV depuis GCS"""
    print(f"Chargement du CSV depuis gs://{bucket_name}/{csv_path}")
    
    client = get_gcs_client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(csv_path)
    
    csv_content = blob.download_as_text()
    
    texts = {}
    reader = csv.DictReader(io.StringIO(csv_content))
    for row in reader:
        texts[row['id']] = row['texte']
    
    print(f"✅ CSV chargé: {len(texts)} textes de référence")
    return texts


def list_audio_files_gcs(bucket_name, prefix, accent):
    """Liste les fichiers audio d'un accent depuis GCS"""
    client = get_gcs_client()
    bucket = client.bucket(bucket_name)
    
    # Construire le chemin complet
    accent_prefix = f"{prefix}{accent}/"
    
    print(f"Recherche dans gs://{bucket_name}/{accent_prefix}")
    
    blobs = bucket.list_blobs(prefix=accent_prefix)
    audio_files = [
        blob.name for blob in blobs 
        if blob.name.endswith('.mp3') and not blob.name.endswith('/')
    ]
    
    return audio_files


def download_audio_from_gcs(bucket_name, blob_path):
    """Télécharge un fichier audio depuis GCS vers un fichier temporaire"""
    client = get_gcs_client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(blob_path)
    
    # Créer un fichier temporaire
    temp_file = tempfile.NamedTemporaryFile(delete=False, suffix='.mp3')
    blob.download_to_filename(temp_file.name)
    
    return temp_file.name

#### CHARGEMENT DES DONNÉES TTS DEPUIS GCS

In [13]:
def load_accent_dataset_gcs(bucket_name, prefix, accent, texts):
    """Charge les données audio pour un accent depuis GCS"""
    
    audio_files = list_audio_files_gcs(bucket_name, prefix, accent)
    
    if not audio_files:
        print(f"Aucun fichier .mp3 trouvé pour {accent}")
        return None
    
    data = []
    errors = 0
    
    print(f"Chargement de {len(audio_files)} fichiers pour {accent}...")
    
    for i, blob_path in enumerate(audio_files):
        try:
            # Extraire l'ID depuis le nom du fichier
            filename = blob_path.split('/')[-1]
            audio_id = filename.split('_')[0]
            
            if audio_id not in texts:
                errors += 1
                continue
            
            # Le chemin GCS complet pour l'audio
            gcs_path = f"gs://{bucket_name}/{blob_path}"
            
            data.append({
                'audio': gcs_path, 
                'text': texts[audio_id],
                'id': audio_id,
                'accent': accent,
                'source': 'tts'
            })
            
            if (i + 1) % 50 == 0:
                print(f"  ✓ {i+1}/{len(audio_files)} fichiers traités")
                
        except Exception as e:
            print(f"Erreur avec {blob_path}: {e}")
            errors += 1
    
    if errors > 0:
        print(f"{errors} fichiers ignorés")
    
    print(f"{accent:12}: {len(data):4} échantillons TTS")
    
    return Dataset.from_list(data) if data else None

In [14]:
def load_tts_dataset_gcs(bucket_name, prefix, csv_path, accents=ACCENTS):
    """Charge et combine tous les accents TTS depuis GCS"""
    
    print(f"\n{'='*70}")
    print(f"  CHARGEMENT DES DONNÉES TTS DEPUIS GCS")
    print(f"{'='*70}\n")
    print(f"Bucket: {bucket_name}")
    print(f"Prefix: {prefix}\n")
    
    texts = load_csv_from_gcs(bucket_name, csv_path)
    
    datasets = []
    for accent in accents:
        ds = load_accent_dataset_gcs(bucket_name, prefix, accent, texts)
        if ds is not None:
            datasets.append(ds)
    
    if not datasets:
        raise ValueError("❌ Aucun dataset TTS chargé !")
    
    combined = concatenate_datasets(datasets)
    combined = combined.shuffle(seed=42)
    
    print(f"\n DATASET TTS COMBINÉ: {len(combined)} échantillons")
    print(f"{'-'*70}")
    
    for accent in accents:
        count = sum(1 for x in combined if x['accent'] == accent)
        if count > 0:
            pct = 100 * count / len(combined)
            print(f"{accent:<15} {count:>10} {pct:>14.1f}%")
    
    return combined


### CHARGEMENT DE COMMON VOICE

In [15]:
MOZILLA_API_KEY = "677a411f05058853a42f53ae66d6612f575443d75ad46918a02f7f255519429d"

In [17]:
# Vérifier la structure
cv_base = "/home/jupyter/common_voice_v23/cv-corpus-23.0-2025-09-05/fr"
print(f"✓ Répertoire existe: {os.path.exists(cv_base)}")
print(f"✓ train.tsv existe: {os.path.exists(os.path.join(cv_base, 'train.tsv'))}")
print(f"✓ clips/ existe: {os.path.exists(os.path.join(cv_base, 'clips'))}")

✓ Répertoire existe: True
✓ train.tsv existe: True
✓ clips/ existe: True


In [18]:
cv_dir = "/home/jupyter/common_voice_v23"
# Trouver le dossier corpus
cv_dirs = [
    d for d in os.listdir(cv_dir)
    if d.startswith('cv-corpus-23') and os.path.isdir(os.path.join(cv_dir, d))
]

corpus_dir = cv_dirs[0]
tsv_path = os.path.join(cv_dir, corpus_dir, 'fr', 'train.tsv')
tsv_path

'/home/jupyter/common_voice_v23/cv-corpus-23.0-2025-09-05/fr/train.tsv'

In [19]:
df = pd.read_csv(tsv_path, sep="\t", low_memory=False)  
df.head()

,client_id,path,sentence_id,sentence,sentence_domain,up_votes,down_votes,age,gender,accents,variant,locale,segment
0,c6571ecb8c3d8e1f7b3178e86aa39483e3ac7834518ef8...,common_voice_fr_23625905.mp3,231d18b7e779ec7a40d54e3099ce966177671b7e40b240...,Mademoiselle Saget haussa les épaules.,NaN,2,0,NaN,NaN,NaN,NaN,fr,NaN
1,c6571ecb8c3d8e1f7b3178e86aa39483e3ac7834518ef8...,common_voice_fr_23625906.mp3,231336c9d8113bf1e5b93027a21b799dab645b783b0838...,"Affaiblis et dorénavant à terre, Saga, Camus e...",NaN,2,1,NaN,NaN,NaN,NaN,fr,NaN
2,c6571ecb8c3d8e1f7b3178e86aa39483e3ac7834518ef8...,common_voice_fr_23625907.mp3,232d2370edb1f1b37396c49b7c5f4b5cb4956adfec3d43...,"Il adopte un plan de masse rectangulaire, mais...",NaN,2,0,NaN,NaN,NaN,NaN,fr,NaN
3,c6571ecb8c3d8e1f7b3178e86aa39483e3ac7834518ef8...,common_voice_fr_23625908.mp3,2320cb458d79b7ffd979549b252a492c58dbb75851e7d9...,Il est le cousin du rabbin Abraham Meyer.,NaN,2,0,NaN,NaN,NaN,NaN,fr,NaN
4,c6571ecb8c3d8e1f7b3178e86aa39483e3ac7834518ef8...,common_voice_fr_23625909.mp3,232b1824972cfc2bf34f365916f75ca2cec242c01d2485...,Il se trouvait en apparence dans une boîte de ...,NaN,2,0,NaN,NaN,NaN,NaN,fr,NaN


In [20]:
df['age'].nunique()

8

In [21]:
df.nunique()

client_id            9535
path               598539
sentence_id        598539
sentence           598539
sentence_domain        15
up_votes               11
down_votes              7
age                     8
gender                  4
accents               133
variant                 0
locale                  1
segment                 0
dtype: int64

In [22]:
gender_counts = df['gender'].value_counts(dropna=False)
gender_counts

gender
male_masculine        344920
NaN                   183306
female_feminine        70021
do_not_wish_to_say       263
non-binary                29
Name: count, dtype: int64

#### Gestion des NaN dans le dataset Common Voice

In [23]:
def load_common_voice_mozilla_v23(cv_dir, split='train', max_samples=None):
    """Charge Common Voice avec gestion correcte des NaN"""
    print(f"\n{'='*70}")
    print(f"   CHARGEMENT COMMON VOICE 23.0 - {split.upper()}")
    print(f"{'='*70}\n")
    
    # Trouver le dossier corpus
    cv_dirs = [
        d for d in os.listdir(cv_dir)
        if d.startswith('cv-corpus-23') and os.path.isdir(os.path.join(cv_dir, d))
    ]
    
    if not cv_dirs:
        raise ValueError(f"Aucun dossier cv-corpus-23.* trouvé dans {cv_dir}")
    
    corpus_dir = cv_dirs[0]
    corpus_path = os.path.join(cv_dir, corpus_dir)
    print(f" Corpus détecté : {corpus_dir}")
    
    fr_dir = os.path.join(corpus_path, 'fr')
    if not os.path.exists(fr_dir):
        raise ValueError(f"Dossier français introuvable : {fr_dir}")
    
    tsv_path = os.path.join(fr_dir, f"{split}.tsv")
    if not os.path.exists(tsv_path):
        raise ValueError(f"Fichier TSV introuvable : {tsv_path}")
    
    print(f" Lecture de {tsv_path}...")
    df = pd.read_csv(tsv_path, sep="\t", low_memory=False)
    print(f"{len(df)} échantillons dans le split {split}")
    
    # Échantillonner si demandé
    if max_samples and len(df) > max_samples:
        df = df.sample(n=max_samples, random_state=42)
        print(f" Échantillonnage réduit à {max_samples} lignes")
    
    clips_dir = os.path.join(fr_dir, "clips")
    data = []
    missing = 0
    
    for idx, row in df.iterrows():
        audio_path = os.path.join(clips_dir, row["path"])
        
        if not os.path.exists(audio_path):
            missing += 1
            continue
        
        # Remplacement simplifié des NaN par "unknown"
        age_str = "unknown" if pd.isna(row.get("age")) else str(row["age"])
        gender_str = "unknown" if pd.isna(row.get("gender")) else str(row["gender"])
        
        data.append({
            "audio": audio_path,
            "text": str(row["sentence"]),
            "id": str(row["path"]).replace(".mp3", ""),
            "accent": "common_voice",
            "source": "common_voice_v23",
            "age": age_str,
            "gender": gender_str,
        })
        
        if (idx + 1) % 1000 == 0:
            print(f"   Traités : {idx+1}/{len(df)}")
    
    if missing > 0:
        print(f"  Fichiers manquants : {missing}")
    
    print(f"{len(data)} échantillons valides chargés\n")
    
    return Dataset.from_list(data)

### CHARGEMENT DES DONNÉES FRANCOFLEX ET UTILISATEURS\n
\n
Intègre les datasets suivants pour réduire le WER:\n
- **VoxForge French**: ~60 locuteurs, audio WAV 16kHz\n
- **Mozilla CV deltas**: 12-2025, 03-2025, 06-2025 (accents variés)\n
- **data_users.csv**: Conversations utilisateurs (phrases françaises)\n
\n
Normalisation appliquée:\n
- Correction encodage VoxForge (é→é)\n
- Conversion en minuscules\n
- Filtrage échantillons validés (up_votes >= 2)

In [ ]:
import re
import ast

# =============================================================================
# NORMALISATION DU TEXTE POUR FRANCOFLEX
# =============================================================================

# Corrections d'encodage pour VoxForge (Latin-1 interprété comme UTF-8)
CORRECTIONS_ENCODAGE = {
    'é': 'é', 'è': 'è', 'ê': 'ê', 'ë': 'ë',
    'à': 'à', 'â': 'â', 'ä': 'ä',
    'ù': 'ù', 'û': 'û', 'ü': 'ü',
    'ô': 'ô', 'ö': 'ö',
    'î': 'î', 'ï': 'ï',
    'ç': 'ç',
    'œ': 'œ', 'æ': 'æ',
    'Ã©': 'é', 'Ã¨': 'è', 'Ãª': 'ê',
    'Ã ': 'à', 'Ã¢': 'â',
    'Ã¹': 'ù', 'Ã»': 'û',
    'Ã´': 'ô', 'Ã®': 'î', 'Ã§': 'ç',
}

# Correspondance des accents Mozilla CV vers labels standardisés
CORRESPONDANCE_ACCENTS = {
    'Français de France': 'france',
    'Français du Canada': 'canada',
    'Français du Québec': 'canada',
    'Français de Belgique': 'belgique',
    'Français de Suisse': 'suisse',
    'Français de Madagascar': 'madagascar',
    'Français du sud de France': 'france',
}


def corriger_encodage(texte):
    """Corrige les problèmes d'encodage hérités de VoxForge"""
    for incorrect, correct in CORRECTIONS_ENCODAGE.items():
        texte = texte.replace(incorrect, correct)
    return texte


def normaliser_texte_francais(texte, depuis_voxforge=False):
    """
    Normalise le texte français pour l'entraînement Whisper.
    - Convertit en minuscules (attendu par Whisper)
    - Corrige les problèmes d'encodage
    - Supprime les espaces superflus
    """
    if depuis_voxforge:
        texte = corriger_encodage(texte)
    
    texte = texte.lower()
    texte = re.sub(r'\s+', ' ', texte).strip()
    texte = ''.join(c for c in texte if c.isprintable() or c == ' ')
    
    return texte


def mapper_accent(chaine_accent):
    """Mappe les chaînes d'accent Mozilla CV vers des labels standardisés"""
    if pd.isna(chaine_accent) or not chaine_accent:
        return 'inconnu'
    
    for cle, valeur in CORRESPONDANCE_ACCENTS.items():
        if cle in str(chaine_accent):
            return valeur
    
    return 'inconnu'


def charger_donnees_utilisateurs_gcs(nom_bucket, chemin_csv, max_echantillons=2000):
    """
    Charge les données de conversation utilisateur depuis GCS.
    Extrait les phrases de l'utilisateur (rôle 'user') comme données d'entraînement texte.
    
    Note: Ces données n'ont pas d'audio associé - utilisées pour augmentation textuelle.
    
    Args:
        nom_bucket: Nom du bucket GCS
        chemin_csv: Chemin vers data_users.csv dans GCS
        max_echantillons: Nombre maximum d'échantillons à charger
    
    Returns:
        Liste de dictionnaires avec les phrases utilisateur normalisées
    """
    print(f"\n{'='*70}")
    print(f"   CHARGEMENT DONNÉES UTILISATEURS DEPUIS GCS")
    print(f"{'='*70}\n")
    
    client = get_gcs_client()
    bucket = client.bucket(nom_bucket)
    blob = bucket.blob(chemin_csv)
    
    if not blob.exists():
        print(f"   Fichier non trouvé: gs://{nom_bucket}/{chemin_csv}")
        return []
    
    # Télécharger et lire le CSV
    contenu_csv = blob.download_as_text()
    df = pd.read_csv(io.StringIO(contenu_csv), low_memory=False)
    
    print(f"   Lignes dans le CSV: {len(df)}")
    
    donnees_texte = []
    compteur = 0
    
    for idx, ligne in df.iterrows():
        if compteur >= max_echantillons:
            break
        
        # Extraire la colonne conversation (format JSON/liste Python)
        conversation = ligne.get('conversation', '')
        if pd.isna(conversation) or not conversation:
            continue
        
        try:
            # Parser la conversation (c'est une chaîne représentant une liste)
            if isinstance(conversation, str):
                messages = ast.literal_eval(conversation)
            else:
                messages = conversation
            
            # Extraire uniquement les messages de l'utilisateur
            for msg in messages:
                if msg.get('role') == 'user':
                    texte = msg.get('content', '').strip()
                    
                    # Ignorer les messages trop courts ou vides
                    if len(texte) < 5:
                        continue
                    
                    # Normaliser le texte
                    texte_normalise = normaliser_texte_francais(texte)
                    
                    if texte_normalise:
                        donnees_texte.append({
                            'texte': texte_normalise,
                            'id_utilisateur': str(ligne.get('userId', 'inconnu')),
                            'niveau': str(ligne.get('userLevel', 'inconnu')),
                            'langue_source': str(ligne.get('userLanguage', 'inconnu')),
                        })
                        compteur += 1
                        
                        if compteur >= max_echantillons:
                            break
        
        except Exception as e:
            continue  # Ignorer les lignes mal formatées
    
    print(f"   Phrases utilisateur extraites: {len(donnees_texte)}")
    
    return donnees_texte


def charger_francoflex_depuis_gcs(nom_bucket, prefixe_francoflex, max_echantillons=5000):
    """
    Charge les données Francoflex (VoxForge + Mozilla CV deltas) depuis GCS.
    
    Parcourt l'intégralité du répertoire VoxForgeFr/ pour récupérer:
    - Transcriptions: gs://bucket/francoflex/VoxForgeFr/{locuteur}/etc/PROMPTS
    - Audio: gs://bucket/francoflex/VoxForgeFr/{locuteur}/wav/*.wav
    
    Structure GCS attendue:
    gs://bucket/francoflex/
        VoxForgeFr/
            Locuteur-Date-xxx/
                wav/*.wav
                etc/PROMPTS
        MozillaCV/
            Dataset 12-2025/cv-corpus-.../fr/validated.tsv, clips/
            Dataset 03-2025/...
            Dataset 06-2025/...
    
    Args:
        nom_bucket: Nom du bucket GCS
        prefixe_francoflex: Préfixe du dossier Francoflex dans GCS
        max_echantillons: Nombre maximum d'échantillons à charger
    
    Returns:
        Dataset HuggingFace combiné
    """
    print(f"\n{'='*70}")
    print(f"   CHARGEMENT FRANCOFLEX DEPUIS GCS")
    print(f"{'='*70}\n")
    
    client = get_gcs_client()
    bucket = client.bucket(nom_bucket)
    
    donnees = []
    
    # =========================================================================
    # 1. CHARGER VOXFORGE - Parcourir tout VoxForgeFr/
    # =========================================================================
    print(" Chargement de VoxForge French...")
    prefixe_voxforge = f"{prefixe_francoflex}VoxForgeFr/"
    
    # Lister tous les blobs sous VoxForgeFr/
    blobs_voxforge = list(bucket.list_blobs(prefix=prefixe_voxforge))
    blobs_prompts = [b for b in blobs_voxforge if b.name.endswith('/etc/PROMPTS')]
    
    print(f"   Fichiers PROMPTS trouvés: {len(blobs_prompts)}")
    
    compteur_voxforge = 0
    for blob_prompts in blobs_prompts:
        try:
            # Essayer UTF-8 d'abord, puis Latin-1 en fallback
            try:
                contenu_prompts = blob_prompts.download_as_text(encoding='utf-8')
            except Exception:
                contenu_prompts = blob_prompts.download_as_text(encoding='latin-1')
            
            # Obtenir le répertoire du locuteur
            rep_locuteur = blob_prompts.name.split('/')[-3]
            prefixe_wav = blob_prompts.name.replace('/etc/PROMPTS', '/wav/')
            
            lignes = contenu_prompts.strip().split('\n')
            
            for ligne in lignes:
                if not ligne.strip():
                    continue
                
                parties = ligne.split(' ', 1)
                if len(parties) != 2:
                    continue
                
                id_fichier = parties[0].split('/')[-1]  # ex: "fr-sb-763"
                transcription = parties[1]
                
                # Construire le chemin audio GCS
                chemin_audio_gcs = f"gs://{nom_bucket}/{prefixe_wav}{id_fichier}.wav"
                
                # Normaliser le texte (corriger encodage + minuscules)
                texte_normalise = normaliser_texte_francais(transcription, depuis_voxforge=True)
                
                donnees.append({
                    'audio': chemin_audio_gcs,
                    'text': texte_normalise,
                    'id': f"voxforge_{rep_locuteur}_{id_fichier}",
                    'accent': 'france',
                    'source': 'voxforge'
                })
                compteur_voxforge += 1
        
        except Exception as e:
            print(f"   Erreur {blob_prompts.name}: {e}")
    
    print(f"   Échantillons VoxForge chargés: {compteur_voxforge}")
    
    # =========================================================================
    # 2. CHARGER LES DELTAS MOZILLA CV (depuis MozillaCV/, pas VoxForgeFr/)
    # =========================================================================
    print("\n Chargement des deltas Mozilla Common Voice...")
    
    # CORRECTION: Lister MozillaCV/ séparément
    prefixe_mozilla = f"{prefixe_francoflex}MozillaCV/"
    blobs_mozilla = list(bucket.list_blobs(prefix=prefixe_mozilla))
    blobs_tsv = [b for b in blobs_mozilla if b.name.endswith('validated.tsv')]
    
    print(f"   Fichiers validated.tsv trouvés: {len(blobs_tsv)}")
    
    echantillons_par_delta = max(1, (max_echantillons - compteur_voxforge) // max(1, len(blobs_tsv)))
    
    for blob_tsv in blobs_tsv:
        try:
            print(f"   Traitement de: {blob_tsv.name}")
            
            # Télécharger le TSV
            contenu_tsv = blob_tsv.download_as_text()
            
            # Parser avec pandas
            df = pd.read_csv(io.StringIO(contenu_tsv), sep='\t', low_memory=False)
            
            # Filtrer par votes positifs
            df = df[df['up_votes'] >= 2]
            
            # Échantillonner si nécessaire
            if len(df) > echantillons_par_delta:
                df = df.sample(n=echantillons_par_delta, random_state=42)
            
            # Obtenir le répertoire des clips
            rep_tsv = '/'.join(blob_tsv.name.split('/')[:-1])
            prefixe_clips = f"{rep_tsv}/clips/"
            
            compteur_delta = 0
            for _, ligne in df.iterrows():
                chemin_audio_gcs = f"gs://{nom_bucket}/{prefixe_clips}{ligne['path']}"
                
                # Normaliser le texte
                texte_normalise = normaliser_texte_francais(str(ligne['sentence']))
                
                donnees.append({
                    'audio': chemin_audio_gcs,
                    'text': texte_normalise,
                    'id': f"francoflex_cv_{ligne['path'].replace('.mp3', '')}",
                    'accent': mapper_accent(ligne.get('accents', '')),
                    'source': 'francoflex_cv'
                })
                compteur_delta += 1
            
            print(f"      Chargés: {compteur_delta} échantillons")
        
        except Exception as e:
            print(f"   Erreur {blob_tsv.name}: {e}")
    
    # Limiter le nombre total d'échantillons
    if len(donnees) > max_echantillons:
        import random
        random.seed(42)
        donnees = random.sample(donnees, max_echantillons)
    
    print(f"\n   TOTAL ÉCHANTILLONS FRANCOFLEX: {len(donnees)}")
    
    return Dataset.from_list(donnees) if donnees else None

#### PRÉPARATION POUR WHISPER

In [24]:
def combiner_tous_les_datasets(dataset_tts, dataset_cv, dataset_francoflex, config):
    """Combine tous les datasets : TTS + Common Voice + Francoflex"""
    print(f"\n{'='*70}")
    print(f"   COMBINAISON DE TOUS LES DATASETS")
    print(f"{'='*70}\n")
    
    datasets_a_combiner = [dataset_tts]
    print(f"   TTS:          {len(dataset_tts):>6} échantillons")
    
    if config.get('use_common_voice') and dataset_cv is not None and len(dataset_cv) > 0:
        datasets_a_combiner.append(dataset_cv)
        print(f"   Common Voice: {len(dataset_cv):>6} échantillons")
    
    if config.get('use_francoflex') and dataset_francoflex is not None and len(dataset_francoflex) > 0:
        datasets_a_combiner.append(dataset_francoflex)
        print(f"   Francoflex:   {len(dataset_francoflex):>6} échantillons")
    
    combine = concatenate_datasets(datasets_a_combiner).shuffle(seed=42)
    total = len(combine)
    print(f"\n   TOTAL COMBINÉ: {total} échantillons")
    
    # Répartition par source
    sources = {}
    for item in combine:
        src = item.get('source', 'inconnu')
        sources[src] = sources.get(src, 0) + 1
    
    for src, compte in sorted(sources.items()):
        pct = 100 * compte / total
        print(f"      {src:<20} {compte:>6} ({pct:.1f}%)")
    
    return combine

In [25]:
class WhisperDataPreprocessor:
    def __init__(self, model_name, project_id):
        self.processor = WhisperProcessor.from_pretrained(model_name)
        self.project_id = project_id
    
    def _load_audio(self, audio_path):
        # Chargement depuis GCS
        if audio_path.startswith("gs://"):
            client = storage.Client(project=self.project_id)
            bucket_name, blob_path = audio_path[5:].split("/", 1)
            blob = client.bucket(bucket_name).blob(blob_path).download_as_bytes()
            audio, sr = sf.read(io.BytesIO(blob))
        # Chargement local
        else:
            audio, sr = sf.read(audio_path)
        
        # Stéréo vers mono
        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)
        
        # Rééchantillonnage vers 16 kHz
        if sr != 16000:
            target_len = int(len(audio) * 16000 / sr)
            audio = resample(audio, target_len)
        
        return audio.astype(np.float32)
    
    def prepare_dataset(self, batch):
        """Prépare un échantillon pour l'entraînement Whisper"""
        audio = self._load_audio(batch["audio"])
        
        # Extraction des caractéristiques audio
        input_features = self.processor.feature_extractor(
            audio, sampling_rate=16000
        ).input_features[0]
        
        # Tokenisation du texte
        labels = self.processor.tokenizer(
            batch["text"],
            max_length=225,
            truncation=True
        ).input_ids
        
        return {
            "input_features": input_features,
            "labels": labels
        }

    
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int = None
    
    @staticmethod
    def shift_tokens_right(labels: torch.Tensor, pad_token_id: int, decoder_start_token_id: int) -> torch.Tensor:
        """Décale les tokens vers la droite pour créer decoder_input_ids"""
        shifted_input_ids = labels.new_zeros(labels.shape)
        shifted_input_ids[:, 1:] = labels[:, :-1].clone()
        shifted_input_ids[:, 0] = decoder_start_token_id
        shifted_input_ids.masked_fill_(shifted_input_ids == -100, pad_token_id)
        return shifted_input_ids
    
    def __call__(self, features):
        # 1. Extraire les caractéristiques audio
        input_features = []
        for f in features:
            feat = f["input_features"]
            if not isinstance(feat, np.ndarray):
                feat = np.array(feat)
            input_features.append({"input_features": feat})
        
        # 2. Padder l'audio
        batch = self.processor.feature_extractor.pad(
            input_features, 
            return_tensors="pt"
        )
        
        # 3. Padder les labels
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )
        
        # 4. Remplacer le padding par -100 pour le calcul de la perte
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        
        if self.decoder_start_token_id is None:
             raise ValueError("decoder_start_token_id doit être fourni pour créer decoder_input_ids.")
                
        decoder_input_ids = self.shift_tokens_right(
            labels_batch["input_ids"],
            pad_token_id=self.processor.tokenizer.pad_token_id,
            decoder_start_token_id=self.decoder_start_token_id,
        )
        
        batch["labels"] = labels
        batch["decoder_input_ids"] = decoder_input_ids
        
        return batch

In [26]:
def split_dataset(dataset, train_ratio=0.8, val_ratio=0.1):
    """Divise en train/val/test"""
    n = len(dataset)
    train_size = int(n * train_ratio)
    val_size = int(n * val_ratio)
    
    train_ds = dataset.select(range(train_size))
    val_ds = dataset.select(range(train_size, train_size + val_size))
    test_ds = dataset.select(range(train_size + val_size, n))
    
    print(f" Split:")
    print(f"   Train: {len(train_ds):4} ({100*train_ratio:.0f}%)")
    print(f"   Val:   {len(val_ds):4} ({100*val_ratio:.0f}%)")
    print(f"   Test:  {len(test_ds):4} ({100*(1-train_ratio-val_ratio):.0f}%)\n")
    
    return train_ds, val_ds, test_ds

In [27]:
def verify_dataset_format(train_ds):
    """Vérifie que les données sont au bon format"""
    print("\n" + "="*70)
    print("   VÉRIFICATION FORMAT DATASET")
    print("="*70)
    
    example = train_ds[0]
    
    print(f"\nKeys: {list(example.keys())}")
    print(f"\n1. input_features:")
    print(f"   Type: {type(example['input_features'])}")
    
    if isinstance(example['input_features'], np.ndarray):
        print(f"   ✅ Format: numpy.ndarray")
        print(f"   Shape: {example['input_features'].shape}")
    elif isinstance(example['input_features'], list):
        print(f"   ❌ Format: list (PROBLÈME!)")
        print(f"   Length: {len(example['input_features'])}")
        # Convertir pour voir
        arr = np.array(example['input_features'])
        print(f"   Après conversion: {arr.shape}")
    else:
        print(f"   ⚠️ Format inconnu: {type(example['input_features'])}")
    
    print(f"\n2. labels:")
    print(f"   Type: {type(example['labels'])}")
    print(f"   Length: {len(example['labels'])}")
    
    if len(example['labels']) == 0:
        print(f"   ❌ Labels vides!")
    else:
        print(f"   ✅ Labels OK")
        print(f"   Sample: {example['labels'][:5]}...")
    
    print("="*70 + "\n")

In [28]:
def log_memory_usage(message):
    # RAM SYSTEME
    process = psutil.Process(os.getpid())
    ram_usage = process.memory_info().rss / (1024 ** 3) # en Gigaoctets
    total_ram = psutil.virtual_memory().total / (1024 ** 3)
    
    # VRAM (si disponible)
    try:
        gpus = GPU.getGPUs()
        if gpus:
            gpu = gpus[0]
            vram_usage = gpu.memoryUsed / 1024 # en Gigaoctets
            vram_total = gpu.memoryTotal / 1024
            vram_info = f" | VRAM GPU: {vram_usage:.2f}/{vram_total:.2f} GB"
        else:
            vram_info = ""
    except Exception:
        vram_info = ""
    
    print(f"\n{message}:")
    print(f"   RAM Processus: {ram_usage:.2f} GB / {total_ram:.2f} GB{vram_info}")

#### ENTRAÎNEMENT

In [29]:
def telecharger_modele_vers_gcs(rep_local, nom_bucket, prefixe_gcs):
    """Télécharge le modèle entraîné vers GCS"""
    client = storage.Client(project=GCS_CONFIG['project_id'])
    bucket = client.bucket(nom_bucket)
    
    for racine, reps, fichiers in os.walk(rep_local):
        for fichier in fichiers:
            chemin_local = os.path.join(racine, fichier)
            chemin_relatif = os.path.relpath(chemin_local, rep_local)
            chemin_gcs = f"{prefixe_gcs}{chemin_relatif}"
            
            blob = bucket.blob(chemin_gcs)
            blob.upload_from_filename(chemin_local)
            print(f"  ✓ {chemin_gcs}")
    
    print(f"\n✅ Modèle téléchargé dans gs://{nom_bucket}/{prefixe_gcs}")


def entrainer_whisper_vertex_ai(config=CONFIG, gcs_config=GCS_CONFIG):
    """
    Fonction principale d'entraînement Whisper sur Vertex AI.
    Charge tous les datasets, prétraite les données et lance l'entraînement.
    """
    print(f"\n{'='*70}")
    print(f"   ENTRAÎNEMENT WHISPER SUR VERTEX AI")
    print(f"{'='*70}\n")
    
    # 1. CHARGER TTS
    dataset_tts = load_tts_dataset_gcs(
        gcs_config['bucket_name'],
        gcs_config['tts_prefix'],
        gcs_config['csv_file']
    )
    
    # 2. CHARGER COMMON VOICE
    dataset_cv = None
    if config.get('use_common_voice', False):
        rep_cv = "/home/jupyter/common_voice_v23"
        if os.path.exists(rep_cv):
            dataset_cv = load_common_voice_mozilla_v23(
                cv_dir=rep_cv,
                split='train',
                max_samples=config.get('common_voice_max_samples', 10000)
            )
    
    # 3. CHARGER FRANCOFLEX (NOUVEAU!)
    dataset_francoflex = None
    if config.get('use_francoflex', False):
        dataset_francoflex = charger_francoflex_depuis_gcs(
            nom_bucket=gcs_config['bucket_name'],
            prefixe_francoflex='francoflex/',
            max_echantillons=config.get('francoflex_max_samples', 5000)
        )
    
    # 4. CHARGER DONNÉES UTILISATEURS (pour enrichissement textuel)
    if config.get('use_data_users', False):
        donnees_utilisateurs = charger_donnees_utilisateurs_gcs(
            nom_bucket=gcs_config['bucket_name'],
            chemin_csv=config.get('data_users_csv', 'francoflex_raw_data/data_users.csv'),
            max_echantillons=config.get('data_users_max_samples', 2000)
        )
        print(f"   Données utilisateurs chargées: {len(donnees_utilisateurs)} phrases")
        # Note: Ces données textuelles peuvent être utilisées pour augmentation future
    
    # 5. COMBINER TOUS LES DATASETS
    dataset_complet = combiner_tous_les_datasets(
        dataset_tts,
        dataset_cv,
        dataset_francoflex,
        config
    )
    
    # 6. DIVISER EN TRAIN/VAL/TEST
    train_ds, val_ds, test_ds = split_dataset(dataset_complet)
    
    # 7. PRÉPROCESSEUR
    print(f"🔧 Initialisation du préprocesseur...")
    preprocesseur = WhisperDataPreprocessor(
        model_name=config['model_name'],
        project_id=gcs_config['project_id']
    )
    log_memory_usage("📌 MÉMOIRE AVANT PRÉTRAITEMENT")
    
    print(f"📊 Prétraitement des données...")
    train_ds = train_ds.map(
        preprocesseur.prepare_dataset,
        remove_columns=train_ds.column_names,
        num_proc=4,
        desc="Train",
        writer_batch_size=500
    )
    
    val_ds = val_ds.map(
        preprocesseur.prepare_dataset,
        remove_columns=val_ds.column_names,
        num_proc=4,
        desc="Val",
        writer_batch_size=500
    )
    
    # 8. MODÈLE
    print(f"\n🤖 Chargement du modèle...")
    model = WhisperForConditionalGeneration.from_pretrained(config['model_name'])
    
    model.config.use_cache = False
    print("✅ Gradient checkpointing activé (économise 40% RAM)")
    
    if 'dropout' in config:
        model.config.dropout = config['dropout']
        model.config.attention_dropout = config['dropout']
        model.config.activation_dropout = config['dropout']
        print(f"✅ Dropout configuré à {config['dropout']}")
    
    # FORCER LE FRANÇAIS
    print(f"🇫🇷 Configuration pour le français...")
    model.generation_config.language = "french"
    model.generation_config.task = "transcribe"
    model.config.suppress_tokens = []
    print(f"✅ Modèle configuré pour le français")
    
    # 9. COLLATEUR DE DONNÉES
    data_collator = DataCollatorSpeechSeq2SeqWithPadding(
        processor=preprocesseur.processor,
        decoder_start_token_id=model.config.decoder_start_token_id,
    )
    
    # 10. ARGUMENTS D'ENTRAÎNEMENT
    training_args = Seq2SeqTrainingArguments(
        output_dir="./whisper-output",
        per_device_train_batch_size=config['batch_size'],
        gradient_accumulation_steps=config['gradient_accumulation_steps'],
        learning_rate=config['learning_rate'],
        warmup_steps=config['warmup_steps'],
        num_train_epochs=config['num_epochs'],
        eval_strategy="steps",
        eval_steps=config['eval_steps'],
        save_strategy="steps",
        save_steps=config['save_steps'],
        logging_steps=10,
        torch_empty_cache_steps=10,
        
        load_best_model_at_end=True,
        metric_for_best_model="wer",
        greater_is_better=False,
        save_total_limit=config['save_total_limit'],
        label_smoothing_factor=config.get('label_smoothing', 0.0),
        
        fp16=False,
        bf16=torch.cuda.is_available(),
        gradient_checkpointing=False,
        
        predict_with_generate=True,
        generation_max_length=128,
    )
    
    # 11. MÉTRIQUES
    wer_metric = evaluate.load("wer")
    cer_metric = evaluate.load("cer")
    
    def calculer_metriques(pred):
        pred_ids = pred.predictions
        label_ids = pred.label_ids
        label_ids[label_ids == -100] = preprocesseur.processor.tokenizer.pad_token_id
        
        pred_str = preprocesseur.processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
        label_str = preprocesseur.processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
        
        wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
        cer = 100 * cer_metric.compute(predictions=pred_str, references=label_str)
        
        return {"wer": wer, "cer": cer}
    
    # 12. ENTRAÎNEUR
    trainer = Seq2SeqTrainer(
        args=training_args,
        model=model,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=data_collator,
        compute_metrics=calculer_metriques,
        processing_class=preprocesseur.processor,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=config['early_stopping_patience']
            )
        ],
    )
    
    print(f"\n{'='*70}")
    print(f"   DÉBUT DE L'ENTRAÎNEMENT")
    print(f"{'='*70}\n")
    
    trainer.train()
    
    # SAUVEGARDER
    chemin_modele = "./whisper-tts-commonvoice-francoflex"
    trainer.save_model(chemin_modele)
    preprocesseur.processor.save_pretrained(chemin_modele)
    
    print(f"\n✅ Modèle sauvegardé: {chemin_modele}")
    
    # TÉLÉCHARGER VERS GCS
    print(f"\n📤 Téléchargement vers GCS...")
    telecharger_modele_vers_gcs(chemin_modele, gcs_config['bucket_name'], gcs_config['output_prefix'])
    
    return trainer, test_ds, preprocesseur


In [30]:
# Nettoyer
torch.cuda.empty_cache()
gc.collect()

# Afficher l'état mémoire
if torch.cuda.is_available():
    print(f" GPU disponible: {torch.cuda.get_device_name(0)}")
    total_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f" Mémoire totale: {total_memory:.2f} GB")
    
    allocated = torch.cuda.memory_allocated(0) / 1024**3
    reserved = torch.cuda.memory_reserved(0) / 1024**3
    print(f" Mémoire utilisée: {allocated:.2f} GB")
    print(f" Mémoire réservée: {reserved:.2f} GB")
    print(f" Mémoire libre: {total_memory - reserved:.2f} GB\n")

🔋 GPU disponible: Tesla T4
📊 Mémoire totale: 14.57 GB
📊 Mémoire utilisée: 0.00 GB
📊 Mémoire réservée: 0.00 GB
📊 Mémoire libre: 14.57 GB



#### MAIN

In [31]:
# Affichage de la configuration
print(f"   CONFIGURATION")
print(f"{'='*70}\n")
print(f"📦 Bucket: {GCS_CONFIG['bucket_name']}")
print(f"🎤 TTS: {GCS_CONFIG['tts_prefix']}")
print(f"💾 Sortie: gs://{GCS_CONFIG['bucket_name']}/{GCS_CONFIG['output_prefix']}")
print(f"🗣️ Common Voice: {'✅ Activé' if CONFIG['use_common_voice'] else '❌ Désactivé'}")
print(f"🇫🇷 Francoflex: {'✅ Activé' if CONFIG.get('use_francoflex') else '❌ Désactivé'}")
print(f"👤 Données utilisateurs: {'✅ Activé' if CONFIG.get('use_data_users') else '❌ Désactivé'}\n")

# Lancer l'entraînement
trainer, test_ds, preprocesseur = entrainer_whisper_vertex_ai()

print(f"\n✅ Entraînement terminé !")

#### Extraction Common Voice